In [7]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
import warnings
from dataclasses import dataclass
from typing import Optional

import numpy as np
import torch
from PIL import Image

from fundus_image_toolbox.utils import ImageTorchUtils as Img

warnings.simplefilter("always", UserWarning)
np.set_printoptions(suppress=True)

In [9]:
@dataclass
class TensorExpectation:
    expected_shape: Optional[tuple] = None
    expected_dtype: Optional[torch.dtype] = None
    expected_ndim: Optional[int] = None
    note: str = ""


def _shape_of(x):
    return tuple(x.shape) if hasattr(x, "shape") else None


def _dtype_of(x):
    return getattr(x, "dtype", None)


def _check_expectations(case_name: str, out, expectation: Optional[TensorExpectation]):
    if expectation is None:
        return

    problems = []
    out_shape = _shape_of(out)
    out_dtype = _dtype_of(out)
    out_ndim = len(out_shape) if out_shape is not None else None

    if expectation.expected_shape is not None and out_shape != expectation.expected_shape:
        problems.append(f"shape expected {expectation.expected_shape}, got {out_shape}")
    if expectation.expected_dtype is not None and out_dtype != expectation.expected_dtype:
        problems.append(f"dtype expected {expectation.expected_dtype}, got {out_dtype}")
    if expectation.expected_ndim is not None and out_ndim != expectation.expected_ndim:
        problems.append(f"ndim expected {expectation.expected_ndim}, got {out_ndim}")

    if problems:
        warnings.warn(f"[{case_name}] expectation mismatch: {'; '.join(problems)}", UserWarning)


def run_case(case_name: str, obj, *, img_ndims: int = 3, expectation: Optional[TensorExpectation] = None):
    print(f"\n=== {case_name} ===")
    print(f"input type={type(obj)} shape={_shape_of(obj)} dtype={_dtype_of(obj)} img_ndims={img_ndims}")

    try:
        out = Img(obj).to_tensor(img_ndims=img_ndims).img
        print(f"output type={type(out)} shape={_shape_of(out)} dtype={_dtype_of(out)}")
        _check_expectations(case_name, out, expectation)
        return out
    except Exception as exc:
        warnings.warn(f"[{case_name}] exception: {type(exc).__name__}: {exc}", UserWarning)
        return None

In [10]:
rng = np.random.default_rng(0)
H, W = 6, 8

img_2d_u8 = rng.integers(0, 256, size=(H, W), dtype=np.uint8)
img_2d_f32 = rng.random((H, W), dtype=np.float32)

img_hwc_u8 = rng.integers(0, 256, size=(H, W, 3), dtype=np.uint8)
img_hwc_f32 = rng.random((H, W, 3), dtype=np.float32)

img_1hw_u8 = rng.integers(0, 256, size=(1, H, W), dtype=np.uint8)
img_chw_u8 = rng.integers(0, 256, size=(3, H, W), dtype=np.uint8)

pil_gray = Image.fromarray(img_2d_u8)#, mode="L")
pil_rgb = Image.fromarray(img_hwc_u8)#, mode="RGB")

torch_chw = torch.rand(3, H, W)
torch_hw = torch.rand(H, W)

batch_list_2d = [img_2d_u8, img_2d_u8.copy()]
batch_array_bhw = np.stack(batch_list_2d, axis=0)
batch_list_hwc = [img_hwc_u8, img_hwc_u8.copy()]
batch_array_bhwc = np.stack(batch_list_hwc, axis=0)
batch_list_1hw = [img_1hw_u8, img_1hw_u8.copy()]

In [11]:
# Single-image cases
run_case(
    "single_np_2d_uint8",
    img_2d_u8,
    img_ndims=2,
    expectation=TensorExpectation(expected_shape=(1, H, W), expected_dtype=torch.float32, expected_ndim=3),
)
run_case(
    "single_np_2d_float32",
    img_2d_f32,
    img_ndims=2,
    expectation=TensorExpectation(expected_shape=(1, H, W), expected_dtype=torch.float32, expected_ndim=3),
)
run_case(
    "single_np_hwc_uint8",
    img_hwc_u8,
    img_ndims=3,
    expectation=TensorExpectation(expected_shape=(3, H, W), expected_dtype=torch.float32, expected_ndim=3),
)
run_case(
    "single_np_hwc_float32",
    img_hwc_f32,
    img_ndims=3,
    expectation=TensorExpectation(expected_shape=(3, H, W), expected_dtype=torch.float32, expected_ndim=3),
)
run_case(
    "single_np_1hw_uint8_with_ndims2",
    img_1hw_u8,
    img_ndims=2,
    expectation=TensorExpectation(expected_shape=(1, H, W), expected_dtype=torch.float32, expected_ndim=3),
)
run_case(
    "single_np_chw_uint8_with_ndims3",
    img_chw_u8,
    img_ndims=3,
    expectation=TensorExpectation(expected_shape=(3, H, W), expected_dtype=torch.float32, expected_ndim=3),
)
run_case(
    "single_pil_gray",
    pil_gray,
    img_ndims=2,
    expectation=TensorExpectation(expected_shape=(1, H, W), expected_dtype=torch.float32, expected_ndim=3),
)
run_case(
    "single_pil_rgb",
    pil_rgb,
    img_ndims=3,
    expectation=TensorExpectation(expected_shape=(3, H, W), expected_dtype=torch.float32, expected_ndim=3),
)
run_case(
    "single_torch_chw",
    torch_chw,
    img_ndims=3,
    expectation=TensorExpectation(expected_shape=(3, H, W), expected_dtype=torch.float32, expected_ndim=3),
)
run_case(
    "single_torch_hw",
    torch_hw,
    img_ndims=2,
    expectation=TensorExpectation(expected_shape=(1, H, W), expected_dtype=torch.float32, expected_ndim=3),
)


=== single_np_2d_uint8 ===
input type=<class 'numpy.ndarray'> shape=(6, 8) dtype=uint8 img_ndims=2
output type=<class 'torch.Tensor'> shape=(1, 6, 8) dtype=torch.float32

=== single_np_2d_float32 ===
input type=<class 'numpy.ndarray'> shape=(6, 8) dtype=float32 img_ndims=2
output type=<class 'torch.Tensor'> shape=(1, 6, 8) dtype=torch.float32

=== single_np_hwc_uint8 ===
input type=<class 'numpy.ndarray'> shape=(6, 8, 3) dtype=uint8 img_ndims=3
color branch in to_tensor, got image with shape (6, 8, 3)
output type=<class 'torch.Tensor'> shape=(3, 6, 8) dtype=torch.float32

=== single_np_hwc_float32 ===
input type=<class 'numpy.ndarray'> shape=(6, 8, 3) dtype=float32 img_ndims=3
color branch in to_tensor, got image with shape (6, 8, 3)
output type=<class 'torch.Tensor'> shape=(3, 6, 8) dtype=torch.float32

=== single_np_1hw_uint8_with_ndims2 ===
input type=<class 'numpy.ndarray'> shape=(1, 6, 8) dtype=uint8 img_ndims=2
output type=<class 'torch.Tensor'> shape=(1, 6, 8) dtype=torch.float

tensor([[[0.6709, 0.9959, 0.3004, 0.5867, 0.5477, 0.7510, 0.1743, 0.3048],
         [0.7346, 0.8455, 0.0313, 0.8636, 0.6768, 0.0568, 0.9196, 0.8873],
         [0.1053, 0.6300, 0.9023, 0.0952, 0.0935, 0.3407, 0.7262, 0.2766],
         [0.0872, 0.6502, 0.5461, 0.4076, 0.8474, 0.1877, 0.6496, 0.8375],
         [0.5495, 0.0583, 0.4781, 0.9629, 0.9602, 0.2589, 0.4175, 0.8965],
         [0.9562, 0.2072, 0.0448, 0.0763, 0.2411, 0.9750, 0.0807, 0.5242]]])

In [13]:
# Batch-like cases
run_case(
    "batch_list_2d_ndims2",
    batch_list_2d,
    img_ndims=2,
    expectation=TensorExpectation(expected_shape=(2, 1, H, W), expected_dtype=torch.float32, expected_ndim=4),
)
run_case(
    "batch_array_bhw_ndims2",
    batch_array_bhw,
    img_ndims=2,
    expectation=TensorExpectation(expected_shape=(2, 1, H, W), expected_dtype=torch.float32, expected_ndim=4),
)
run_case(
    "batch_array_bhw_ndims3_intentional_mismatch",
    batch_array_bhw,
    img_ndims=3,
    expectation=TensorExpectation(expected_shape=(2, 3, H, W), expected_dtype=torch.float32, expected_ndim=4),
)
run_case(
    "batch_list_hwc_ndims3",
    batch_list_hwc,
    img_ndims=3,
    expectation=TensorExpectation(expected_shape=(2, 3, H, W), expected_dtype=torch.float32, expected_ndim=4),
)
run_case(
    "batch_array_bhwc_ndims3",
    batch_array_bhwc,
    img_ndims=3,
    expectation=TensorExpectation(expected_shape=(2, 3, H, W), expected_dtype=torch.float32, expected_ndim=4),
)
run_case(
    "batch_list_1hw_ndims2",
    batch_list_1hw,
    img_ndims=2,
    expectation=TensorExpectation(expected_shape=(2, 1, H, W), expected_dtype=torch.float32, expected_ndim=4),
)


=== batch_list_2d_ndims2 ===
input type=<class 'list'> shape=None dtype=None img_ndims=2
You passed a batch-like object to to_tensor. In this case, from_cspace and to_cspace are ignored for batch images. Use to_cspace instead.
output type=<class 'torch.Tensor'> shape=(2, 1, 6, 8) dtype=torch.float32

=== batch_array_bhw_ndims2 ===
input type=<class 'numpy.ndarray'> shape=(2, 6, 8) dtype=uint8 img_ndims=2
You passed a batch-like object to to_tensor. In this case, from_cspace and to_cspace are ignored for batch images. Use to_cspace instead.
output type=<class 'torch.Tensor'> shape=(2, 1, 6, 8) dtype=torch.float32

=== batch_array_bhw_ndims3_intentional_mismatch ===
input type=<class 'numpy.ndarray'> shape=(2, 6, 8) dtype=uint8 img_ndims=3
color branch in to_tensor, got image with shape (2, 6, 8)

=== batch_list_hwc_ndims3 ===
input type=<class 'list'> shape=None dtype=None img_ndims=3
You passed a batch-like object to to_tensor. In this case, from_cspace and to_cspace are ignored for b

/tmp/ipykernel_3188767/3838559830.py:47: UserWarning: [batch_array_bhw_ndims3_intentional_mismatch] exception: TypeError: Cannot handle this data type: (1, 1, 8), |u1
  warnings.warn(f"[{case_name}] exception: {type(exc).__name__}: {exc}", UserWarning)


tensor([[[[0.2627, 0.6118, 0.4431, 0.6510, 0.8863, 0.6980, 0.1412, 0.3451],
          [0.9294, 0.2039, 0.9137, 0.2235, 0.7725, 0.0549, 0.1569, 0.4314],
          [0.2431, 0.0314, 0.5647, 0.8745, 0.1961, 0.8471, 0.3098, 0.9686],
          [0.5922, 0.1804, 0.1098, 0.1412, 0.0588, 0.4275, 0.9333, 0.5608],
          [0.2353, 0.6118, 0.9451, 0.7608, 0.1922, 0.9529, 0.2667, 0.2588],
          [0.4980, 0.9137, 0.8000, 0.2706, 0.6588, 0.4588, 0.8706, 0.2392]]],


        [[[0.2627, 0.6118, 0.4431, 0.6510, 0.8863, 0.6980, 0.1412, 0.3451],
          [0.9294, 0.2039, 0.9137, 0.2235, 0.7725, 0.0549, 0.1569, 0.4314],
          [0.2431, 0.0314, 0.5647, 0.8745, 0.1961, 0.8471, 0.3098, 0.9686],
          [0.5922, 0.1804, 0.1098, 0.1412, 0.0588, 0.4275, 0.9333, 0.5608],
          [0.2353, 0.6118, 0.9451, 0.7608, 0.1922, 0.9529, 0.2667, 0.2588],
          [0.4980, 0.9137, 0.8000, 0.2706, 0.6588, 0.4588, 0.8706, 0.2392]]]])